# Deploying the model with Gradio

Part two of the deployment exercise. This notebook loads the classifier trained in
[model.ipynb](./model.ipynb) and wraps it in a [Gradio](https://gradio.app/) interface, so
it can be used from a browser by someone who has never opened a notebook.

**Prerequisite:** run `model.ipynb` first to produce `model.pkl` in this folder. It is not
checked into the repository.

The `#|export` comments mark the cells that belong to the standalone app rather than to
the exploratory session. The last cell in this notebook collects them into `gradio.py` —
the same idea as fastai's `nbdev`, in about fifteen lines.

`is_cat` has to be redefined here even though it is not called: the pickled learner
references it by name, so it must exist when `load_learner` unpickles the file.

In [ ]:
#|export
from fastai.vision.all import *
import gradio as gr 

def is_cat(x): return x[0].isupper()

## Try it on one image

A quick manual check with the upload widget, to confirm the model loads and predicts
sensibly before any interface is built around it.

In [ ]:
import ipywidgets as widgets
uploader = widgets.FileUpload()
uploader

In [ ]:
img = PILImage.create(uploader.data[0])
img.thumbnail((192,192))
img

## Load the exported learner

`load_learner` restores the trained model from `model.pkl`. Nothing about the training
data or the training code is needed here — that is the whole point of exporting.

In [ ]:
#|export 
learn = load_learner("model.pkl")

In [ ]:
learn.predict(img)

## Wrap prediction for Gradio

Gradio expects a plain function. `classify_image` takes an image, runs the model, and
returns a mapping of category to probability, which Gradio renders as a labelled bar chart.

In [ ]:
#|export
categories = ("Dog", "Cat")

def classify_image(img):
    pred,idx,probs = learn.predict(img)
    return dict(zip(categories, map(float,probs)))

In [ ]:
classify_image(img)

## Build the interface

An image input, a label output, and three example images so a visitor can try it without
finding a picture of their own. `dunno.jpg` is deliberately neither a cat nor a dog — it
shows how the model behaves when asked something outside what it was trained on.

In [ ]:
#|export
image = gr.Image()
label = gr.Label()

In [ ]:
#|export
examples = ["dog.jpg", "cat.jpg", "dunno.jpg"]

intf = gr.Interface(fn=classify_image, inputs=image, outputs=label, examples=examples)
intf.launch(inline=False)

## Export the app to a script

Collect every cell marked `#|export` into `gradio.py`, which can then be run as a normal
Python program or deployed to somewhere like Hugging Face Spaces.

In [ ]:
import json

with open('gradio.ipynb', 'r') as f:
    nb = json.load(f)

with open('gradio.py', 'w') as out:
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            source = cell['source']
            if source and (source[0].strip().startswith('#|export') or source[0].strip().startswith('#|export')):
                for line in source[1:]:
                    out.write(line)
                out.write('\n\n')

print("Exported cells with #|export to gradio.py")
